# 2일차 파트 2 실습: 구조화 출력과 시스템 연동

실습은 세 개입니다(실습 5, 6, 7). 접수 요청 80건을 담당팀, 위치, 긴급 여부로 정리해 DB에 저장하고, 결과를 얼마나 믿을 수 있는지 숫자로 확인합니다.
`NotImplementedError`가 있는 곳을 직접 채웁니다. 막히면 힌트를 펼쳐 보고, 다 한 뒤에 `2일차_02_실습_풀이.ipynb`와 비교합니다.
테스트 셀은 mock 호출을 써서 여러분의 코드가 맞게 동작하는지 알려 줍니다. API를 부르지 않으므로 요금이 들지 않습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

일자마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(2일차 파트 2의 "구조화 출력" 절) |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 일자의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

# 설치: 파트 3의 예제는 python -m pip install sentence-transformers scikit-learn도 필요합니다
DATA = 'data/2일차'
SYSTEM = open(f'{DATA}/분류_규칙.txt', encoding='utf-8').read()

# 100만 토큰당 달러 (입력, 출력). Anthropic 공개 가격, 2026-11-03 확인.
# 소형은 입력이 10만 토큰 이하인 요청의 단가입니다. 넘으면 (0.50, 2.50)입니다.
# 사내 API를 쓴다면 사내 과금 기준으로 바꿉니다. 과금이 없으면 (0, 0)으로 둡니다.
PRICE = {'소형': (0.10, 0.50), '기본': (2.00, 10.00), '대형': (4.00, 20.00)}


def cost_usd(input_tokens, output_tokens, model='기본'):
    """토큰 수와 단가로 요금(달러)을 계산합니다."""
    input_price, output_price = PRICE[model]
    return (input_tokens * input_price + output_tokens * output_price) / 1_000_000


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 실습 준비

아래 셀은 세 실습이 함께 쓰는 재료와 함수입니다. 그대로 실행합니다.

| 이름 | 내용 |
| --- | --- |
| `tickets` | 접수 80건. 열은 `id`, `접수시각`, `본문`입니다 |
| `ticket_answers` | 정답 라벨. 열은 `team`, `location`, `urgent`, `난이도`입니다. `난이도`에는 판단하기 어려운 엣지 케이스 15건이 `엣지`, 나머지 65건이 `보통`으로 적혀 있습니다. 위치가 없는 접수의 `location`은 `None`입니다 |
| `TICKET_RULES` | 담당팀 기준(`접수_규칙.txt`). 시스템 프롬프트로 씁니다 |
| `prompt_of(row)` | 접수 한 건을 모델에 보낼 글로 만듭니다 |
| `parse_reply(reply, model_class)` | 응답을 검증해 `(객체, None)` 또는 `(None, 사유)`를 돌려줍니다. 교재의 예제 2-5와 같습니다 |
| `open_db(path)` | `requests` 테이블이 있는 SQLite DB를 엽니다. `id`가 기본 키입니다 |

In [ ]:
# 실습 준비
import sqlite3
from typing import Literal

from pydantic import BaseModel, ConfigDict, ValidationError

import checks2

tickets = pd.read_csv(f'{DATA}/접수_실습.csv').fillna('')
ticket_answers = pd.read_csv(f'{DATA}/접수_실습_정답.csv', dtype=str, keep_default_na=False).set_index('id')
ticket_answers['urgent'] = ticket_answers['urgent'] == 'true'
ticket_answers['location'] = ticket_answers['location'].map(lambda value: value or None)
TICKET_RULES = open(f'{DATA}/접수_규칙.txt', encoding='utf-8').read().strip()
print(TICKET_RULES)
print('----')
print(tickets.head(8).to_string(index=False))


def prompt_of(row):
    """접수 한 건을 모델에 보낼 글로 만듭니다."""
    return f"id: {row['id']}\n본문: {row['본문']}"


def parse_reply(reply, model_class):
    """응답을 검증해 (객체, None) 또는 (None, 사유)를 돌려줍니다."""
    if reply.stop_reason != 'end':
        return None, {'max_tokens': '잘림', 'refusal': '거절'}.get(reply.stop_reason, '비정상 종료')
    text = reply.text[reply.text.find('{'):reply.text.rfind('}') + 1]
    try:
        return model_class.model_validate_json(text), None
    except ValidationError as error:
        first = error.errors()[0]
        field = '.'.join(str(part) for part in first['loc']) or '전체'
        return None, f"스키마 불일치({field}: {first['type']})"


def open_db(path=':memory:'):
    """requests 테이블이 있는 DB를 엽니다. 경로를 주지 않으면 메모리에만 있는 DB입니다."""
    db = sqlite3.connect(path)
    db.execute('''CREATE TABLE IF NOT EXISTS requests (
        id TEXT PRIMARY KEY, team TEXT, location TEXT, urgent INTEGER,
        model TEXT, run_at TEXT, reviewed TEXT)''')
    return db

## 실습 5 스키마 설계

**목적** 결과 테이블의 열에서 출발해 스키마를 정합니다. 허용값과, 값이 없을 때의 처리를 직접 정해 봅니다.

**할 일** pydantic 클래스 `Ticket`을 작성합니다. 결과 테이블에 필요한 열은 네 개입니다.

| 필드 | 내용 |
| --- | --- |
| `id` | 접수 id |
| `team` | 담당팀. 설비, 시설, IT, 안전, 기타 가운데 하나 |
| `location` | 본문에 나온 장비 이름이나 장소. 본문에 없으면 값이 없습니다 |
| `urgent` | 긴급 여부 |

**완료 기준** 테스트 셀이 "실습 5 통과"를 출력합니다. 그다음 3건을 실제로 보내, 위치가 없는 접수의 `location`이 `None`으로 오는지 확인합니다.

<details><summary>힌트 1</summary>

허용값은 `Literal['설비', '시설', ...]`로, 없을 수 있는 값은 `str | None`으로 적습니다.
</details>

<details><summary>힌트 2</summary>

`location: str | None = None`처럼 기본값을 주면 그 필드는 필수 필드에서 빠집니다. 구조화 출력의 스키마는 모든 필드가 필수여야 하므로 기본값을 주지 않습니다. 값이 없다는 것은 필드를 빼는 것이 아니라 `null`을 넣는 것으로 표현합니다.
</details>

<details><summary>힌트 3</summary>

정하지 않은 필드를 거부하고 문자열을 불리언 값으로 바꿔 받지 않게 하려면 `model_config = ConfigDict(extra='forbid', strict=True)`를 클래스 안에 적습니다.
</details>

In [ ]:
# 실습 5
class Ticket(BaseModel):
    """접수 한 건의 결과. 필드는 id, team, location, urgent입니다."""
    raise NotImplementedError('이 부분을 채웁니다')

In [ ]:
# 실습 5 확인 (API를 부르지 않습니다)
checks2.check_schema(Ticket)

In [ ]:
# 실습 5 실행 (실제 API를 부릅니다. 3건)
for _, row in tickets.iloc[[3, 5, 7]].iterrows():
    reply = llm.ask(prompt_of(row), system=TICKET_RULES, model='소형', schema=Ticket.model_json_schema())
    parsed, reason = parse_reply(reply, Ticket)
    print(row['본문'][:40])
    print(f'  → {parsed}, 사유: {reason}')

**적어 보기** 3건의 결과에서 고칠 점이 보였다면 무엇을 고쳤는지 적습니다.

- (여기에 적습니다)

## 실습 6 배치 처리와 멱등한 저장

**목적** 한 건의 실패가 배치를 멈추지 않게 하고, 다시 실행해도 행이 중복되지 않으며 이미 처리한 건을 다시 보내지 않게 합니다.

**할 일** 함수 세 개를 작성합니다.

1. `classify(row, ask=llm.ask, model_class=None, rules=None)`: 접수 한 건을 보내고 dict를 돌려줍니다. 어떤 경우에도 예외를 발생시키지 않습니다.
    - `ask(prompt_of(row), system=규칙, model='소형', max_tokens=1024, schema=스키마)`로 보냅니다. `model_class`가 `None`이면 `Ticket`을, `rules`가 `None`이면 `TICKET_RULES`를 씁니다(실습 7의 실험에서 다른 값을 넘깁니다).
    - 돌려주는 dict의 키는 `id`, `answer_id`, `team`, `location`, `urgent`, `error`, `input_tokens`, `output_tokens`입니다. `id`는 보낸 id, `answer_id`는 모델이 돌려준 id입니다.
    - 성공하면 `error`는 `None`입니다. 호출 실패(`llm.CallError`)나 `parse_reply()`가 돌려준 사유가 있으면 `error`에 적고 `team`, `location`, `urgent`, `answer_id`는 `None`으로 둡니다.
2. `save(db, records, model, run_at)`: `error`가 없는 결과만 `requests` 테이블에 넣고 넣은 건수를 돌려줍니다. 같은 `id`가 있으면 `team`, `location`, `urgent`, `model`, `run_at`만 고칩니다. `reviewed` 열은 건드리지 않습니다.
3. `run(db, rows, ask=llm.ask)`: `requests` 테이블에 없는 `id`의 행만 `classify()`로 보내고 `save()`로 저장합니다. `(보낸 건수, 실패 목록)`을 돌려줍니다. 실패 목록은 `[(id, 사유), ...]`입니다.

**완료 기준** 테스트 셀이 "실습 6 통과"를 출력합니다. 그다음 80건을 실제로 처리해 저장한 행 수를 확인하고, 한 번 더 실행했을 때 보내는 건이 0건인지 확인합니다.

<details><summary>힌트 1</summary>

`classify()`는 결과 dict를 먼저 만들어 두고(`error`와 결과 필드는 `None`), 단계마다 채워 나가면 어느 경로에서든 같은 키를 가진 dict를 돌려줄 수 있습니다. `except`에는 `llm.CallError`만 적습니다.
</details>

<details><summary>힌트 2</summary>

`save()`의 SQL은 `INSERT INTO requests (id, team, location, urgent, model, run_at) VALUES (?, ?, ?, ?, ?, ?) ON CONFLICT(id) DO UPDATE SET team = excluded.team, ...`처럼 작성합니다. `INSERT OR REPLACE`는 기존 행을 지우고 새 행을 넣으므로 `reviewed` 열의 값이 남지 않아 마지막 테스트를 통과하지 못합니다.
</details>

<details><summary>힌트 3</summary>

`run()`은 먼저 `SELECT id FROM requests`로 저장된 id의 집합을 만들고, 거기에 없는 행만 골라 보냅니다. 여러 건은 `ThreadPoolExecutor`로 병렬 처리하되, DB에 쓰는 일은 병렬 처리가 끝난 뒤 한 번에 합니다.
</details>

In [ ]:
# 실습 6
def classify(row, ask=llm.ask, model_class=None, rules=None):
    """접수 한 건을 보내 결과 dict를 돌려줍니다. 실패해도 예외를 발생시키지 않습니다."""
    raise NotImplementedError('이 부분을 채웁니다')


def save(db, records, model, run_at):
    """error가 없는 결과를 requests 테이블에 넣고 넣은 건수를 돌려줍니다. 같은 id는 결과 열만 고칩니다."""
    raise NotImplementedError('이 부분을 채웁니다')


def run(db, rows, ask=llm.ask):
    """requests 테이블에 없는 id만 보내고 저장합니다. (보낸 건수, 실패 목록)을 돌려줍니다."""
    raise NotImplementedError('이 부분을 채웁니다')

In [ ]:
# 실습 6 확인 (mock 호출을 씁니다)
checks2.check_batch(classify, save, run, open_db)

In [ ]:
# 실습 6 실행 (실제 API를 부릅니다. 80건)
os.makedirs('out', exist_ok=True)
if os.path.exists('out/requests.db'):
    os.remove('out/requests.db')              # 처음부터 다시 해 볼 수 있게 지웁니다
db = open_db('out/requests.db')
rows = [row for _, row in tickets.iterrows()]

sent, failed = run(db, rows)
print(f'첫 실행: 보낸 건 {sent}, 실패 {failed}')
print('저장한 행', db.execute('SELECT COUNT(*) FROM requests').fetchone()[0])
sent, failed = run(db, rows)
print(f'다시 실행: 보낸 건 {sent}, 실패 {failed}')
print('저장한 행', db.execute('SELECT COUNT(*) FROM requests').fetchone()[0])
print(db.execute('SELECT team, COUNT(*) FROM requests GROUP BY team ORDER BY team').fetchall())
db.close()

## 실습 7 검증, 상태, 정확도

**목적** 스키마 검증을 통과한 결과를 얼마나 믿을 수 있는지 숫자로 확인합니다. 그리고 허용값과 기준 문장이 결과를 어떻게 바꾸는지 실험합니다.

**할 일 1** 함수 두 개를 작성합니다. `record`는 `classify()`가 돌려준 dict이고 `text`는 그 접수의 본문입니다.

1. `validate(record, text)`: 걸린 사유(문자열)를, 통과하면 `None`을 돌려줍니다. 아래 순서로 검사합니다.
    - `error`가 있으면 그 값을 그대로 돌려줍니다.
    - 보낸 `id`와 모델이 돌려준 `answer_id`가 다르면 걸립니다.
    - `team`이 "기타"인데 `urgent`가 `True`이면 걸립니다. 네 팀의 일이 아닌 글이 긴급일 수는 없습니다.
    - `location`이 `None`이 아닌데 본문에 그 글자가 없으면 걸립니다. 규칙은 위치를 "본문의 글자 그대로" 옮기라고 했습니다.
2. `status_of(record, text)`: `error`가 있으면 `'manual'`, 그 밖의 사유로 걸리면 `'review'`, 통과하면 `'auto'`를 돌려줍니다.

**할 일 2** 측정 셀을 실행해 실험 전 실행의 숫자를 확인합니다.

**할 일 3** 실험 A와 실험 B의 셀을 실행하고, 바뀐 숫자와 그 이유를 적습니다.

**완료 기준** 테스트 셀이 "실습 7 통과"를 출력합니다. 스키마 검증 통과율, 업무 규칙에 걸린 건수, 담당팀 정확도, `auto` 가운데 틀린 건수를 적습니다. 두 실험에서 바뀐 숫자와 이유를 한 문장씩 적습니다.

<details><summary>힌트 1</summary>

본문에 그 글자가 있는지는 `record['location'] in text`로 확인합니다. `None`에는 `in`을 쓸 수 없으므로 `None`인지를 먼저 확인합니다.
</details>

<details><summary>힌트 2</summary>

`status_of()`는 `validate()`를 불러 쓰면 됩니다. `error`가 있는 경우만 먼저 가려 `'manual'`로 돌려줍니다.
</details>

In [ ]:
# 실습 7
def validate(record, text):
    """결과 한 건을 검사해 걸린 사유를, 통과하면 None을 돌려줍니다."""
    raise NotImplementedError('이 부분을 채웁니다')


def status_of(record, text):
    """'auto', 'review', 'manual' 가운데 하나를 돌려줍니다."""
    raise NotImplementedError('이 부분을 채웁니다')

In [ ]:
# 실습 7 확인 (API를 부르지 않습니다)
checks2.check_validate(validate, status_of)

In [ ]:
# 실습 7 측정 (실제 API를 부릅니다. 80건)
rows = [row for _, row in tickets.iterrows()]
text_of = dict(zip(tickets['id'], tickets['본문']))


def run_all(model_class=None, rules=None):
    """80건을 모두 분류해 결과 dict의 목록을 돌려줍니다. DB에는 저장하지 않습니다."""
    with ThreadPoolExecutor(max_workers=8) as pool:
        return list(pool.map(lambda row: classify(row, model_class=model_class, rules=rules), rows))


def report(records, label):
    """검증 결과와 정답 라벨 비교를 출력합니다."""
    done = [record for record in records if not record['error']]
    team_ok = sum(record['team'] == ticket_answers.loc[record['id'], 'team'] for record in done)
    urgent_ok = sum(record['urgent'] == ticket_answers.loc[record['id'], 'urgent'] for record in done)
    statuses = [status_of(record, text_of[record['id']]) for record in records]
    reasons = pd.Series([validate(record, text_of[record['id']]) for record in records]).dropna()
    auto_wrong = [record['id'] for record, status in zip(records, statuses)
                  if status == 'auto' and record['team'] != ticket_answers.loc[record['id'], 'team']]
    print(f'[{label}] 스키마 검증 통과 {len(done)}/{len(records)}'
          f', 담당팀 정확도 {team_ok}/{len(done)}, 긴급 여부 정확도 {urgent_ok}/{len(done)}')
    print(f'  상태: {pd.Series(statuses).value_counts().to_dict()}, 걸린 사유: {reasons.value_counts().to_dict()}')
    print(f'  auto {statuses.count("auto")}건 가운데 담당팀이 틀린 건 {len(auto_wrong)}건', auto_wrong)
    urgent_count = sum(bool(record['urgent']) for record in done)
    print(f"  urgent가 true인 건 {urgent_count}건(정답은 {int(ticket_answers['urgent'].sum())}건)")


base = run_all()
report(base, '실험 전')
for record in base:
    answer = ticket_answers.loc[record['id']]
    why = validate(record, text_of[record['id']])
    if why or record['team'] != answer['team']:
        print(f"  {record['id']} 모델 {record['team']} / {record['location']!r}, 정답 {answer['team']}"
              f", 사유 {why}, {text_of[record['id']][:30]}")

In [ ]:
# 실습 7 실험 A (담당팀 허용값에서 IT를 뺍니다. 80건을 다시 보냅니다)
class TicketNoIT(Ticket):
    team: Literal['설비', '시설', '안전', '기타']


no_it = run_all(model_class=TicketNoIT)
report(no_it, 'IT 제외')
it_ids = set(ticket_answers.index[ticket_answers['team'] == 'IT'])
moved = pd.Series([record['team'] for record in no_it if record['id'] in it_ids])
print(f'  정답이 IT인 {len(it_ids)}건의 분류:', moved.value_counts(dropna=False).to_dict())

In [ ]:
# 실습 7 실험 B (규칙에서 긴급 기준 문장을 뺍니다. 80건을 다시 보냅니다)
URGENT_LINE = next(line for line in TICKET_RULES.splitlines() if line.startswith('urgent'))
print('뺀 문장:', URGENT_LINE)
no_rule = run_all(rules=TICKET_RULES.replace(URGENT_LINE, '').strip())
report(no_rule, '긴급 기준 제외')
flipped = [(new, old) for new, old in zip(no_rule, base) if new['urgent'] != old['urgent']]
print('  긴급 판단이 바뀐 건', len(flipped))
for new, old in flipped[:10]:
    print(f"  {new['id']} 실험 전 {old['urgent']} → {new['urgent']}"
          f", 정답 {ticket_answers.loc[new['id'], 'urgent']}, {text_of[new['id']][:34]}")

**적어 보기**

| 항목 | 실험 전 | 실험 A (IT 제외) | 실험 B (긴급 기준 제외) |
| --- | --- | --- | --- |
| 스키마 검증 통과 | | | |
| 업무 규칙에 걸린 건 | | | |
| 담당팀 정확도 | | | |
| 긴급 여부 정확도 | | | |
| `auto` 가운데 담당팀이 틀린 건 | | | |

- 실험 A에서 바뀐 숫자와 이유:
- 실험 B에서 바뀐 숫자와 이유:
- 이 결과 테이블을 담당자에게 넘긴다면 함께 알려 줄 숫자:

## 더 해 보기

- `validate()`에 업무 규칙을 하나 더 넣어 봅니다. 예: 본문에 "멈췄", "대피" 같은 말이 있는데 `urgent`가 `False`인 건을 `review`로 보냅니다. 걸리는 건수와, 그 가운데 실제로 틀린 건수를 비교합니다.
- 담당팀이 틀린 건의 본문을 읽고 `접수_규칙.txt`에 문장을 더해 정확도가 오르는지 확인합니다. 규칙 파일은 사본을 만들어 고칩니다.
- AI 코딩 도구(Claude Code나 opencode)에 "접수 CSV를 읽어 담당팀별로 엑셀 시트를 나누고 DB에 쌓는 프로그램"을 맡겨 봅니다. 실습 6의 테스트(`checks2.check_batch`)가 요구하는 조건을 기준으로 적어 주고, 테스트로 결과를 판정합니다.